# Demo B — Variants of the autoencoder and anomaly detection

**Class 11 · B1–B2** (slides: `slides/11_autoencoders.md`)

* **B1:** denoising and sparse autoencoders on the digits: what the noise or the L1 penalty changes in the reconstruction of noisy digits and in the code (share of active units).
* **B2:** the reconstruction error as an anomaly score: the autoencoder is trained on normal digits only (0 to 7); digits 8 and 9 are the anomalies. Threshold from the normal validation errors, AUROC, and the same task on the indoor-climate series. Failure case: an autoencoder that reconstructs everything.

In [ ]:
import os

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from faa import demo, plotlib, semisup
from faa.demo import show, slide

demo.setup("11")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
X, y = semisup.digits_flat()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)


def mse(a, b):
    return float(np.mean((np.asarray(a) - b) ** 2))

## Slide: Denoising: Learn to Remove the Noise (B1)

Noise: Gaussian with standard deviation 0.3 added to every pixel, then clipped to $[0, 1]$. The **denoising** autoencoder ($64 \to 32 \to 8 \to 32 \to 64$) sees a new noisy copy of each training digit in every epoch and must output the clean one. The plain autoencoder is trained on the clean digits.
Test: the error against the clean digit, when the input is noisy (3 seeds).

In [ ]:
rng = np.random.default_rng(0)
Xte_noisy = np.clip(Xte + rng.normal(0, 0.3, Xte.shape), 0, 1).astype("float32")
rows, keep = [], {}
for name, noise in (("plain", 0.0), ("denoising", 0.3)):
    errs, errs_clean = [], []
    for seed in range(3):
        ae, enc = semisup.autoencoder(64, 8, hidden=(32,), seed=seed)
        semisup.fit_ae(ae, Xtr, noise=noise, epochs=120)
        errs.append(mse(ae.predict(Xte_noisy, verbose=0), Xte))
        errs_clean.append(mse(ae.predict(Xte, verbose=0), Xte))
        if seed == 0:
            keep[name] = ae
    rows.append(
        {
            "autoencoder": name,
            "mse_noisy_input": np.mean(errs),
            "std": np.std(errs),
            "mse_clean_input": np.mean(errs_clean),
        }
    )
pca = PCA(8).fit(Xtr)
rows.append(
    {
        "autoencoder": "PCA k=8",
        "mse_noisy_input": mse(pca.inverse_transform(pca.transform(Xte_noisy)), Xte),
        "std": 0.0,
        "mse_clean_input": mse(pca.inverse_transform(pca.transform(Xte)), Xte),
    }
)
slide("Denoising: Learn to Remove the Noise")
print(f"error of the noisy digits themselves (no model): {mse(Xte_noisy, Xte):.4f}")
show(pl.DataFrame(rows), 4)

idx = np.array([np.flatnonzero(yte == c)[0] for c in range(8)])
fig, axes = plt.subplots(4, 8, figsize=(2.2, 1.2))
for r, (name, imgs) in enumerate(
    [
        ("clean", Xte[idx]),
        ("noisy", Xte_noisy[idx]),
        ("plain AE", np.asarray(keep["plain"].predict(Xte_noisy[idx], verbose=0))),
        ("denoising", np.asarray(keep["denoising"].predict(Xte_noisy[idx], verbose=0))),
    ]
):
    for c in range(8):
        a = axes[r, c]
        a.imshow(imgs[c].reshape(8, 8), cmap="gray_r", vmin=0, vmax=1)
        a.set_xticks([])
        a.set_yticks([])
        a.grid(False)
        if c == 0:
            a.set_ylabel(name, fontsize=5.6)
fig.subplots_adjust(wspace=0.05, hspace=0.05)
plotlib.save(fig, "b1_denoising")

## Slide: Sparse Codes (B1)

An L1 penalty on the activations of the code ($\lambda \lVert z \rVert_1$) with an *over*complete code ($k = 64$, as large as the input): most units must be zero. Table: penalty, test error, share of code units with $|z| > 0.01$ and mean number of active units per digit.

In [ ]:
rows = []
for l1 in (0.0, 1e-4, 1e-3, 1e-2):
    ae, enc = semisup.autoencoder(64, 64, hidden=(), activation="relu", l1=l1, seed=0)
    semisup.fit_ae(ae, Xtr, epochs=120)
    Z = np.asarray(enc.predict(Xte, verbose=0))
    rows.append(
        {
            "l1": f"{l1:g}",
            "test_mse": mse(ae.predict(Xte, verbose=0), Xte),
            "active_share": float(np.mean(np.abs(Z) > 0.01)),
            "active_per_digit": float((np.abs(Z) > 0.01).sum(1).mean()),
        }
    )
slide("Sparse Codes")
sp = pl.DataFrame(rows)
show(sp, 4)

fig, ax = plotlib.fig("col", 1.6)
ax.plot(range(len(sp)), sp["active_per_digit"], color=BLUE, marker="o", ms=3)
ax.set_xticks(range(len(sp)), list(sp["l1"]))
ax.set_xlabel(r"L1 penalty $\lambda$")
ax.set_ylabel("active units per digit")
plotlib.save(fig, "b1_sparse")

## Slide: Anomalies Are Badly Reconstructed (B2)

Normal = digits 0 to 7, anomalies = 8 and 9 (about 25 % of the test digits). Autoencoder $64 \to 32 \to 4 \to 32 \to 64$ trained on 80 % of the normal training rows; the threshold is the 95th percentile of the errors of the other 20 % of the normal rows (validation). Test: normal test digits and the anomalies.

In [ ]:
normal = ytr < 8
Xn = Xtr[normal]
Xfit, Xval = train_test_split(Xn, test_size=0.2, random_state=0)
ae, enc = semisup.autoencoder(64, 4, hidden=(32,), seed=0)
semisup.fit_ae(ae, Xfit, epochs=150)
e_val = semisup.recon_error(ae, Xval)
e_te = semisup.recon_error(ae, Xte)
is_anom = yte >= 8
thr = float(np.quantile(e_val, 0.95))
pred = e_te > thr
tp, fp = int((pred & is_anom).sum()), int((pred & ~is_anom).sum())
fn, tn = int((~pred & is_anom).sum()), int((~pred & ~is_anom).sum())
slide("Anomalies Are Badly Reconstructed")
print(
    f"normal rows: {len(Xfit)} fit, {len(Xval)} validation; test: {int((~is_anom).sum())} normal, {int(is_anom.sum())} anomalies"
)
print(f"threshold (95th percentile of validation errors) {thr:.4f}; AUROC {roc_auc_score(is_anom, e_te):.3f}")
print(f"TP {tp} FP {fp} FN {fn} TN {tn}; recall {tp / (tp + fn):.3f}; precision {tp / max(1, tp + fp):.3f}")

fig, ax = plotlib.fig("col", 1.7)
bins = np.linspace(0, e_te.max(), 40)
ax.hist(e_te[~is_anom], bins=bins, color=BLUE, alpha=0.7, label="normal (0-7)")
ax.hist(e_te[is_anom], bins=bins, color=RED, alpha=0.7, label="anomaly (8, 9)")
ax.axvline(thr, color="k", ls="--", lw=0.8)
ax.set_xlabel("reconstruction error")
ax.set_ylabel("digits")
ax.legend()
plotlib.save(fig, "b2_error_hist")

## Slide: Autoencoder or PCA as a Detector? (B2)

Same task, same normal rows: AUROC of the reconstruction error of an autoencoder and of PCA for several code sizes (3 seeds for the autoencoder).

In [ ]:
rows = []
for k in (2, 4, 8, 16):
    auc_ae = []
    for seed in range(3):
        a, _ = semisup.autoencoder(64, k, hidden=(32,), seed=seed)
        semisup.fit_ae(a, Xfit, epochs=150)
        auc_ae.append(roc_auc_score(is_anom, semisup.recon_error(a, Xte)))
    p = PCA(k).fit(Xfit)
    auc_pca = roc_auc_score(is_anom, np.mean((p.inverse_transform(p.transform(Xte)) - Xte) ** 2, axis=1))
    rows.append({"k": k, "auroc_ae": np.mean(auc_ae), "std": np.std(auc_ae), "auroc_pca": auc_pca})
det = pl.DataFrame(rows)
slide("Autoencoder or PCA as a Detector?")
show(det)

fig, ax = plotlib.fig("col", 1.6)
ax.errorbar(det["k"], det["auroc_ae"], det["std"], color=ORANGE, marker="s", ms=3, capsize=2, label="autoencoder")
ax.plot(det["k"], det["auroc_pca"], color=BLUE, marker="o", ms=3, label="PCA")
ax.set_xscale("log", base=2)
ax.set_xlabel("code size $k$")
ax.set_ylabel("AUROC")
ax.legend()
plotlib.save(fig, "b2_auroc_k")

## Slide: A Room-Temperature Anomaly (B2)

The indoor-climate series (Class 09/10): hourly values of the first five series (temperatures T1-T5), standardised on the first 70 % of the time; autoencoder $5 \to 3 \to 2 \to 3 \to 5$ (tanh, linear output) on the training part. An anomaly is injected in the last 30 %: a sensor offset of +1.5 standard deviations for 24 h, in 5 windows (a fault that breaks the link between the channels). AUROC of the error, per hour.

In [ ]:
from faa import data

M, cols = data.load_indoor_climate()
M = M.T[:, :5]  # hours x 5 temperatures
ntr = int(0.7 * len(M))
mu, sd = M[:ntr].mean(0), M[:ntr].std(0)
Z = ((M - mu) / sd).astype("float32")
Ztest = Z[ntr:].copy()
lab = np.zeros(len(Ztest), dtype=bool)
rng = np.random.default_rng(0)
for s in rng.choice(len(Ztest) - 24, 5, replace=False):
    Ztest[s : s + 24, 1] += 1.5
    lab[s : s + 24] = True
Ztr = Z[:ntr]
keras.utils.set_random_seed(0)
inp = keras.layers.Input((5,))
h = keras.layers.Dense(3, activation="tanh")(inp)
c = keras.layers.Dense(2, activation="tanh")(h)
h2 = keras.layers.Dense(3, activation="tanh")(c)
out = keras.layers.Dense(5)(h2)
cae = keras.Model(inp, out)
cae.compile(optimizer=keras.optimizers.Adam(5e-3), loss="mse")
cae.fit(Ztr, Ztr, epochs=60, batch_size=64, verbose=0)
err = np.mean((np.asarray(cae.predict(Ztest, verbose=0)) - Ztest) ** 2, axis=1)
p2 = PCA(2).fit(Ztr)
err_p = np.mean((p2.inverse_transform(p2.transform(Ztest)) - Ztest) ** 2, axis=1)
slide("A Room-Temperature Anomaly")
print(
    f"{len(Ztr)} training hours, {len(Ztest)} test hours, {int(lab.sum())} anomalous hours (5 offset episodes of 24 h on {cols[1]})"
)
print(f"mean of the test hours in training standard deviations, per channel: {np.round(Z[ntr:].mean(0), 2).tolist()}")
print(
    f"AUROC of the reconstruction error: autoencoder {roc_auc_score(lab, err):.3f}, PCA (k=2) {roc_auc_score(lab, err_p):.3f}"
)
fig, ax = plotlib.fig("full", 1.4)
ax.plot(err, color=BLUE, lw=0.6)
ax.fill_between(np.arange(len(err)), 0, err.max(), where=lab, color=RED, alpha=0.25, label="sensor offset")
ax.set_xlabel("hour of the test part")
ax.set_ylabel("reconstruction error")
ax.legend(loc="upper right")
plotlib.save(fig, "b2_climate_error")

## Failure case: the autoencoder that reconstructs everything

Same detector with a code as large as the input ($k = 64$, the 32-unit layer removed): it reconstructs 8 and 9 as well as the normal digits.

In [ ]:
rows = []
for name, k, hidden in (("k=4, hidden 32", 4, (32,)), ("k=64, no hidden", 64, ())):
    a, _ = semisup.autoencoder(64, k, hidden=hidden, seed=0)
    semisup.fit_ae(a, Xfit, epochs=150)
    e = semisup.recon_error(a, Xte)
    rows.append(
        {
            "model": name,
            "mse_normal": float(e[~is_anom].mean()),
            "mse_anomaly": float(e[is_anom].mean()),
            "auroc": roc_auc_score(is_anom, e),
        }
    )
slide("The autoencoder that reconstructs everything")
show(pl.DataFrame(rows), 4)